# Starter Notebook

This is a new Python notebook ready for analysis and experimentation.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb as ddb

In [61]:
iris_df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv")
iris_df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [12]:
# register the DataFrame as a table in DuckDB
ddb.register("iris", iris_df)

In [13]:
# inspect the schema
ddb.sql("DESCRIBE iris").fetchdf()

,column_name,column_type,null,key,default,extra
0,sepal.length,DOUBLE,YES,None,None,None
1,sepal.width,DOUBLE,YES,None,None,None
2,petal.length,DOUBLE,YES,None,None,None
3,petal.width,DOUBLE,YES,None,None,None
4,variety,VARCHAR,YES,None,None,None


In [43]:
result = ddb.sql("""
    SELECT variety
    ,AVG("petal.length") AS avg_petal_length
    ,AVG("petal.width") AS avg_petal_width
    ,AVG("sepal.length") AS avg_sepal_length
    ,AVG("sepal.width") AS avg_sepal_width
    FROM iris
    GROUP BY ALL
    ORDER BY 1 ASC

""").fetchdf()
result

,variety,avg_petal_length,avg_petal_width,avg_sepal_length,avg_sepal_width
0,Setosa,1.462,0.246,5.006,3.428
1,Versicolor,4.260,1.326,5.936,2.770
2,Virginica,5.552,2.026,6.588,2.974


In [ ]:
result.loc[result['variety'] == 'Setosa', ['avg_petal_length', 'avg_petal_width']]

,avg_petal_length,avg_petal_width
0,1.462,0.246


In [47]:
customer_data = pd.DataFrame({
    'customer_id': ['1', '2', '3', '4'],
    'name': ['Alice', 'Bob', 'Charlie', 'David'],
    'region': ['North', 'South', 'East', 'West']
})
customer_data

,customer_id,name,region
0,1,Alice,North
1,2,Bob,South
2,3,Charlie,East
3,4,David,West


In [49]:
orders_data = pd.DataFrame({
    'order_id': ['1001', '1002', '1003', '1004','1005','1006'],
    'customer_id': ['1', '2', '1', '3', '2', '4'],
    'order_amount': [250, 150, 300, 200, 180, 220]
})     
orders_data   

,order_id,customer_id,order_amount
0,1001,1,250
1,1002,2,150
2,1003,1,300
3,1004,3,200
4,1005,2,180
5,1006,4,220


In [ ]:
ddb.register("customer_data", customer_data)
ddb.register("orders_data", orders_data)

In [60]:
joined = ddb.sql("""
    SELECT
        c.customer_id,
        c.name,
        c.region,
        count(distinct o.order_id) as total_orders,
        sum(o.order_amount) as total_order_amount
    FROM customer_data c
    INNER JOIN orders_data o
        ON c.customer_id = o.customer_id
    GROUP BY all
    ORDER BY name asc
""").fetchdf()

joined

,customer_id,name,region,total_orders,total_order_amount
0,1,Alice,North,2,550.0
1,2,Bob,South,2,330.0
2,3,Charlie,East,1,200.0
3,4,David,West,1,220.0
